In [1]:
import geopandas as gpd
import pandas as pd

gdf = gpd.read_file('Berney_merge_legende_v7-7.geojson').set_crs('EPSG:2056', allow_override=True).to_crs('EPSG:4326')
gdf = gdf.reset_index()
# fixing page number issues
gdf['page'] = pd.to_numeric(gdf['page'], errors='coerce').astype('Int64')

def remove_etc(use_l:str):
    if pd.isna(use_l):
        return use_l
    if 'etc' in use_l:
        return use_l.replace('etc', '').strip()
    if 'etc.' in use_l:
        return use_l.replace('etc.', '').strip()
    return use_l

def use_list_to_list(use_l:str):
    if pd.isna(use_l):
        return use_l
    use_items = None
    if '|' in use_l:
        use_items = [item.strip() for item in use_l.split('|')]
    elif ' et ' in use_l:
        use_items = [item.strip() for item in use_l.split(' et ')]
    else:   
        use_items = [use_l.strip()]
    final_items = []
    for itm in use_items:
        if ',' in itm:
            sub_items = [sub_item.strip() for sub_item in itm.split(',')]
            final_items.extend(sub_items)
        else:
            final_items.append(remove_etc(itm))
    return [f for f in final_items if f]
# homogenize land use entries between appelation with "|" and "et".
gdf['use'] = gdf['use'].apply(use_list_to_list)
df_wh = pd.read_csv('wh_images.csv')

def format_filename_to_code(filename:str):
    code = filename.replace('PC_1827-1831_Berney_Vol-', '')
    if 'legende' in code:
        code = code.replace('_legende', '')
    code = code.split('_')[-1].replace('.jpg', '')
    if code.isnumeric():
        code = int(code)
    return code

df_wh['code'] = df_wh['filename'].apply(format_filename_to_code)

def find_the_right_filename(code):
    matched = df_wh[df_wh['code'] == code]
    if len(matched) == 1:
        return matched.iloc[0]['filename']
    if len(matched) > 1:
        # return only the one whose has "legende" in the filename (the other one is purely a map)
        for idx, row in matched.iterrows():
            if 'legende' in row['filename']:
                return row['filename']
    return None

gdf['page_filename'] = gdf['page'].apply(find_the_right_filename)

# removes duplicates based on geometry (manually checked, they are duplicates, not different entries pointing to the same geometry)
gdf = gpd.GeoDataFrame(gdf.groupby('geometry').first().drop(columns=['index']).reset_index(), geometry='geometry')

gdf.to_file('Berney_merge_legende_v7-7_formatted_for_timeatlas.geojson', driver='GeoJSON')

In [8]:
unique_use = sorted(gdf['use'].explode().dropna().astype(str).str.strip().unique())
unique_use, len(unique_use)

(['-',
  '1/2 four',
  '1/2 grange',
  '1/2 écurie',
  '2 granges',
  '2 écuries',
  '2/5e de pré',
  '3 écuries',
  '3/5e de champ',
  'abattoir',
  'aisances',
  'allee',
  'allee publique',
  'ancien manège',
  'atelier',
  'atelier de menuisier',
  'avenue',
  'baignoire',
  'bains',
  'basse-cour',
  'basserie',
  'batiment',
  'batiment du poids',
  'batiment du signal',
  'battoir',
  'belvedere',
  'bois',
  'bois de chênes',
  'bois de sapins',
  'bosquet',
  'boucherie',
  'boutique',
  'brasserie',
  'buanderie',
  'buanderie .',
  'built',
  'buissons',
  'bureau de justice et police',
  'bureau du département militaire',
  'bûcher',
  'bûchers',
  'cabinet',
  'caserne',
  'casino',
  'cave',
  'caveau',
  'caves .',
  'chambre',
  'chambre au-dessus',
  "chambre d'école",
  'chambre à lessive',
  'chambre à resserrer',
  'champ',
  'chantier',
  'chapelle',
  'chemin',
  'chenevière',
  'château',
  'cimetière',
  "cimetière d'Ouchy",
  'college',
  'contentieux',
  'corp

In [13]:
import re
import unicodedata


def normalize_use_text(value: str) -> str:
    if pd.isna(value):
        return ""
    txt = str(value).strip().lower()
    txt = "".join(
        c for c in unicodedata.normalize("NFD", txt)
        if unicodedata.category(c) != "Mn"
    )
    txt = re.sub(r"[^a-z0-9\s]", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


agri_patterns = [
    ("etable", [r"\betable\b", r"\betables\b", r"\bporc\b", r"\bporcs\b"]),
    ("ecurie", [r"\becurie\b", r"\becuries\b"]),
    ("grange", [r"\bgrange\b", r"\bgranges\b", r"\bgrance\b"]),
    ("fenil", [r"\bfenil\b", r"\bfenils\b"]),
    ("grenier", [r"\bgrenier\b", r"\bgreniers\b"]),
    ("hangar", [r"\bhangar\b", r"\bhangars\b"]),
    ("champ", [r"\bchamp\b", r"\bchamps\b"]),
    ("pre", [r"\bpre\b", r"\bpres\b"]),
    ("paturage", [r"\bpaturage\b"]),
    ("vigne", [r"\bvigne\b", r"\bvignes\b"]),
    ("verger", [r"\bverger\b", r"\bvergers\b"]),
    ("cheneviere", [r"\bcheneviere\b", r"\bchenevieres\b"]),
    ("moulin", [r"\bmoulin\b", r"\bmoulins\b"]),
    ("pressoir", [r"\bpressoir\b", r"\bpressoirs\b"]),
    ("plantage", [r"\bplantage\b", r"\bplantages\b"]),
    ("place_a_fumier", [r"\bfumier\b"]),
    ("voliere", [r"\bvoliere\b", r"\bvolieres\b"]),
    ("abattoir", [r"\babattoir\b", r"\babattoirs\b", r"\btuerie\b"]),
]


def map_agri_category(use_value: str):
    normalized = normalize_use_text(use_value)
    for category, patterns in agri_patterns:
        if any(re.search(pattern, normalized) for pattern in patterns):
            return category
    return None


features = gdf.copy()
features["feature_id"] = features.index

agri_long = features[["feature_id", "geometry", "use"]].explode("use").copy()
agri_long["use_raw"] = agri_long["use"].astype(str).str.strip()
agri_long["agri_category"] = agri_long["use_raw"].apply(map_agri_category)
agri_long = agri_long[agri_long["agri_category"].notna()].copy()

agri_by_feature = agri_long.groupby("feature_id").agg(
    agri_categories=("agri_category", lambda s: sorted(set(s))),
    use=("use_raw", lambda s: sorted(set(s))),
)

agri_selected = features[features["feature_id"].isin(agri_by_feature.index)].copy()
agri_selected = agri_selected.join(agri_by_feature, on="feature_id", rsuffix="_agri_only")

agri_selected["use_all"] = agri_selected["use"]
agri_selected["use"] = agri_selected["use_agri_only"]
agri_selected = agri_selected.drop(columns=["use_agri_only"])

agri_selected.to_file("Berney_merge_legende_v7-7_agriculture_merged.geojson", driver="GeoJSON")

agri_selected[["feature_id", "use", "use_all", "agri_categories"]].head(), len(agri_selected)

(    feature_id                         use                     use_all  \
 14          14             [pré, pâturage]             [pré, pâturage]   
 16          16                       [pré]                       [pré]   
 17          17                       [pré]                       [pré]   
 23          23                       [pré]                       [pré]   
 32          32  [pré marais soit pâturage]  [pré marais soit pâturage]   
 
     agri_categories  
 14  [paturage, pre]  
 16            [pre]  
 17            [pre]  
 23            [pre]  
 32            [pre]  ,
 3863)

In [4]:
agricole=["étables à porcs", "écuries", "écurie", "volière", "vignes", "vigne", "verger", "verre", "pâturage", "pré", "pressoir", "plantage", "place à fumier", "moulin", "grange", "hangar", "grenier", "fenil", "chenevière", "champ", "abattoir", "ecurie", "Ecurie"] 

In [5]:
use[use["use"].isin(agricole)]['use'].unique()

array(['pré', 'pâturage', 'champ', 'pressoir', 'place à fumier', 'vigne',
       'grange', 'écuries', 'grenier', 'hangar', 'écurie', 'fenil',
       'moulin', 'plantage', 'volière', 'verger', 'vignes',
       'étables à porcs', 'abattoir', 'chenevière'], dtype=object)

In [6]:
use[use['use'].isin(['étables à porcs'])]['main_use']

7316    buanderie
Name: main_use, dtype: object

In [10]:
sorted(agri_long.loc[agri_long['agri_category'] == 'etable', 'use_raw'].unique())[:50]

['étable', 'étable à porcs', 'étables à porcs']

In [15]:
use[use['use'].isin(agricole)].to_file('Berney_merge_legende_v7-7_formatted_for_timeatlas.geojson', driver='GeoJSON')

In [7]:
use_ag = use[use['use'].isin(agricole)].copy()

In [8]:
use_ag['use'].unique()

array(['pré', 'pâturage', 'champ', 'pressoir', 'place à fumier', 'vigne',
       'grange', 'écuries', 'grenier', 'hangar', 'écurie', 'fenil',
       'moulin', 'plantage', 'volière', 'verger', 'vignes',
       'étables à porcs', 'abattoir', 'chenevière'], dtype=object)

In [2]:
from collections import Counter

# ── French use terms → target English categories ──────────────────────────────
# volière is intentionally excluded (deleted per request)
USE_CATEGORY_MAP = {
    # allotments
    "chenevière": "allotments",
    "jardin": "allotments",
    "jardin potager": "allotments",
    "jardin pour le chancelier": "allotments",
    "verger": "allotments",
    # animal_keeping
    "1/2 écurie": "animal_keeping",
    "1/2 grange": "animal_keeping",
    "2 écuries": "animal_keeping",
    "2 granges": "animal_keeping",
    "3 écuries": "animal_keeping",
    "basse-cour": "animal_keeping",
    "deux écuries": "animal_keeping",
    "étable": "animal_keeping",
    "étable à porcs": "animal_keeping",
    "étables à porcs": "animal_keeping",
    "écurie": "animal_keeping",
    "écuries": "animal_keeping",
    "fenil": "animal_keeping",
    "fenils": "animal_keeping",
    "grance": "animal_keeping",   # typo variant of grange
    "grange": "animal_keeping",
    "grange?": "animal_keeping",
    "granges": "animal_keeping",
    "hangar": "animal_keeping",
    "place à fumier": "animal_keeping",
    # basin
    "baignoire": "basin",
    "bains": "basin",
    "réservoir": "basin",
    # farmland
    "3/5e de champ": "farmland",
    "champ": "farmland",
    "grenier": "farmland",
    "terre": "farmland",
    # grass
    "2/5e de pré": "grass",
    "pré": "grass",
    "pré marais soit pâturage": "grass",
    "pâturage": "grass",
    # greenhouse_horti
    "serre": "greenhouse_horti",
    "serre sous la terrasse": "greenhouse_horti",
    # plant_nursery
    "pepinière": "plant_nursery",
    "plantage": "plant_nursery",
    # vineyard
    "pressoir": "vineyard",
    "pressoir .": "vineyard",
    "vigne": "vineyard",
    "vignes": "vineyard",
}


def harmonize_uses(use_list):
    """Map French use terms to English categories; drop volière; return sorted list or None."""
    if not isinstance(use_list, list):
        return None
    cats = set()
    for item in use_list:
        if pd.isna(item):
            continue
        s = str(item).strip()
        if s.lower() in ("volière", "voliere"):
            continue
        cat = USE_CATEGORY_MAP.get(s)
        if cat:
            cats.add(cat)
    return sorted(cats) if cats else None


gdf_agri = gdf.copy()
gdf_agri["use_harmonized"] = gdf_agri["use"].apply(harmonize_uses)

# Keep only parcels with at least one recognised agricultural category
gdf_agri = gdf_agri[gdf_agri["use_harmonized"].notna()].copy()

# use_fr = original French list; use = harmonized English categories (list)
gdf_agri = gdf_agri.rename(columns={"use": "use_fr", "use_harmonized": "use"})

print(f"Agricultural features: {len(gdf_agri)}")
cat_counts = Counter(c for cats in gdf_agri["use"] for c in cats)
for cat, count in sorted(cat_counts.items()):
    print(f"  {cat}: {count}")

gdf_agri.to_file("Berney_1831_agriculture_harmonized.geojson", driver="GeoJSON")
print("Saved → Berney_agriculture_harmonized.geojson")


Agricultural features: 4915
  allotments: 1033
  animal_keeping: 667
  basin: 3
  farmland: 1297
  grass: 1188
  greenhouse_horti: 43
  plant_nursery: 30
  vineyard: 674
Saved → Berney_agriculture_harmonized.geojson
